# Mean-reversion on the synthetic mock engine

This notebook mirrors `mean_reversion.py` but adds inline explanations.
It is fully runnable end-to-end with **no C++ build** — the
`MockBacktestEngine` consumes a synthetic L3-like data stream and routes
every event through the adapter layer that the C++ engine will eventually
share.

**Pipeline:**

```
SyntheticDataAdapter  →  MockBacktestEngine  →  MockEventAdapter  →  Strategy
                                                  Strategy  →  MockOrderAdapter  →  Engine
```


In [ ]:
%matplotlib inline
import logging

from python_wrapper_interface import (
    BacktestRunner,
    MockBacktestEngine,
)
from python_wrapper_interface.examples.mean_reversion import MeanReversion

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

## 1. Construct the engine

Using all defaults gives us a `SyntheticDataAdapter`, a
`MockEventAdapter` (which converts the dirty, C++-style events into
clean `BookUpdate` / `Trade` / `Fill` dataclasses) and a
`MockOrderAdapter` that converts the strategy's `Order` objects back
into the engine's command format.

In [ ]:
engine = MockBacktestEngine(
    instrument_ids=[1001],
    num_events=5_000,
    base_price=1.10,
    volatility=0.20,
    seed=7,
)
runner = BacktestRunner(engine)
strategy = MeanReversion()

## 2. Run the backtest

`BacktestRunner.run()` calls `engine.load(...)`, binds the strategy to a
`StrategyContext`, then drives `engine.step()` until exhaustion.

In [ ]:
result = runner.run(
    strategy,
    data_path=".",
    date_range=("2024-01-01", "2024-01-02"),
)

print(f"total fills: {len(result.fills_df)}")
if not result.pnl_series.empty:
    print(f"final PnL: {result.pnl_series['cumulative_pnl'].iloc[-1]:.4f}")

## 3. Inspect the dataframes

`Result` always has three dataframes: `pnl_series`, `fills_df`, and
`order_log_df`.

In [ ]:
result.fills_df.head()

In [ ]:
result.order_log_df.head()

## 4. Visualize

`result.plot_pnl()` delegates to `MatplotlibVisualizer().plot_pnl(result)`.
For the fills-on-mid overlay we use the visualizer directly so we can
show both plots side-by-side without the default `plt.show()` call.

In [ ]:
from python_wrapper_interface.adapters.viz_adapter import MatplotlibVisualizer

viz = MatplotlibVisualizer(show=False)
fig_pnl = viz.plot_pnl(result)
fig_fills = viz.plot_fills_on_mid(result)